Training of the 1D CNN Model

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import tensorflow
from sklearn.model_selection import train_test_split
import tensorflow as tf
import keras
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report

In [141]:
dataset_path = Path("Training Data")

recordings = []
labels = []

Assigning the classes

In [142]:
classes = {
    "Normal" : 0,
    "Near Fall" : 1,
    "Fall" : 2
}

for folder, label in classes.items():
    
    folder_path = dataset_path / folder
    print(folder_path)

    for file in sorted(folder_path.glob("*.csv")):
        data = pd.read_csv(file)

        recordings.append(data)
        labels.append(label)

Training Data\Normal
Training Data\Near Fall
Training Data\Fall


Check whats in the training data

In [143]:
print(f"Loaded {len(recordings)} recordings")

unique, counts = np.unique(labels, return_counts=True)

display_classes = ['Normal', 'Near Fall', 'Fall']

for label, count in zip(unique, counts):
    print(f'{display_classes[label]}: {count}')

Loaded 10 recordings
Normal: 7
Near Fall: 1
Fall: 2


Create 200 sample windows with 50% overlap

In [144]:
window_size = 200   # 4 seconds at 50Hz
overlap = 100     # 50% overlap

x = []
y = []

for recording, label in zip(recordings, labels):

    # convert dataframe -> numpy
    data = recording.values.astype(np.float32)

    # create sliding windows
    for start in range(0, len(data) - window_size + 1, window_size - overlap):

        window = data[start:start + window_size]

        x.append(window)
        y.append(label)


# convert lists to numpy arrays
x = np.array(x, dtype=np.float32)
y = np.array(y, dtype=np.int32)


print("x shape:", x.shape)
print("y shape:", y.shape)
print("classes:", np.unique(y, return_counts=True))

x shape: (333, 200, 6)
y shape: (333,)
classes: (array([0, 1, 2], dtype=int32), array([262,  14,  57]))


get the test and train split data

In [145]:
x_train, x_validate, y_train, y_validate = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('training dataset shape: ', x_train.shape)
print('testing dataset shape: ', x_validate.shape)

print(np.unique(y_train, return_counts=True))
print(np.unique(x_validate, return_counts=True))

training dataset shape:  (266, 200, 6)
testing dataset shape:  (67, 200, 6)
(array([0, 1, 2], dtype=int32), array([209,  11,  46]))
(array([-535.82764, -507.5073 , -499.75583, ...,  471.19138,  497.13132,
        497.92477], shape=(18207,), dtype=float32), array([1, 1, 1, ..., 2, 1, 1], shape=(18207,)))


obtain the mean and std of the training dataset

In [ ]:
mean = x_train.mean(axis=(0,1))
std = x_train.std(axis=(0,1))

std = np.maximum(std, 1e-6)

x_train = (x_train - mean) / std
x_validate = (x_validate - mean) / std

In [147]:
np.save("mean.npy", mean)
np.save("std.npy", std)

print(mean, std)

[ 0.25478387 -0.82250375  0.06058095 -0.29674637  1.5617149  -0.01077443] [ 0.35722736  0.40110552  0.26830512 34.03168    59.997665   44.646103  ]


In [148]:
print("train:", np.unique(y_train, return_counts=True))
print("test:", np.unique(x_validate, return_counts=True))

train: (array([0, 1, 2], dtype=int32), array([209,  11,  46]))
test: (array([-12.001425, -11.367096, -11.193476, ...,  10.554161,  10.83596 ,
        12.426778], shape=(26773,), dtype=float32), array([1, 1, 1, ..., 2, 1, 1], shape=(26773,)))


train the 1d cnn model

In [149]:
model = models.Sequential([
    layers.Input(shape=(200,6)),

    layers.Conv1D(
        filters=32,
        kernel_size=5,
        activation="relu"
    ),

    layers.MaxPooling1D(pool_size=2),

    layers.Conv1D(
        filters=64,
        kernel_size=5,
        activation="relu"
    ),

    layers.GlobalAveragePooling1D(),

    layers.Dense(64, activation="relu"),

    layers.Dropout(0.3),

    layers.Dense(3, activation="softmax")
])

In [150]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [151]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_validate, y_validate),
    epochs=100,
    batch_size=64,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )
    ]
)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.5977 - loss: 1.0524 - val_accuracy: 0.7910 - val_loss: 0.8660
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7857 - loss: 0.8472 - val_accuracy: 0.7910 - val_loss: 0.7342
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7857 - loss: 0.7228 - val_accuracy: 0.7910 - val_loss: 0.6475
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8045 - loss: 0.6649 - val_accuracy: 0.8358 - val_loss: 0.5650
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8571 - loss: 0.5701 - val_accuracy: 0.8507 - val_loss: 0.4933
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8722 - loss: 0.5277 - val_accuracy: 0.8806 - val_loss: 0.4325
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8835 - loss: 0.4541 - val_accuracy: 0.9104 - val_loss: 0.3740
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9135 - loss: 0.4059 - val_accuracy: 0.9403 - val_loss:

In [152]:
loss, acc = model.evaluate(x_validate, y_validate)

print(acc)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0044 
1.0


Testing the model

In [153]:
test_dataset_path = Path('Test Data')

classes = {
    "Normal" : 0,
    "Near Fall" : 1,
    "Fall" : 2
}

test_recordings = []
test_labels = []

for folder, label in classes.items():
    
    folder_path = test_dataset_path / folder
    print(folder_path)

    for file in sorted(folder_path.glob("*.csv")):
        data = pd.read_csv(file)

        test_recordings.append(data)
        test_labels.append(label)

Test Data\Normal
Test Data\Near Fall
Test Data\Fall


In [154]:
window_size = 200   # 4 seconds at 50Hz
overlap = 100     # 50% overlap

test_x = []
test_y = []

for recording, label in zip(test_recordings, test_labels):

    # convert dataframe -> numpy
    data = recording.values.astype(np.float32)

    # create sliding windows
    for start in range(0, len(data) - window_size + 1, window_size - overlap):

        window = data[start:start + window_size]

        test_x.append(window)
        test_y.append(label)


# convert lists to numpy arrays
test_x = np.array(test_x, dtype=np.float32)
test_y = np.array(test_y, dtype=np.int32)


print("x shape:", test_x.shape)
print("y shape:", test_y.shape)
print("classes:", np.unique(test_y, return_counts=True))

x shape: (136, 200, 6)
y shape: (136,)
classes: (array([0, 1, 2], dtype=int32), array([108,   2,  26]))


In [157]:
test_x = (test_x - mean) / std

In [ ]:
y_prob = model.predict(test_x, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(confusion_matrix(test_y, y_pred))

print(
    classification_report(
        test_y,
        y_pred,
        target_names=["Normal", "Near Fall", "Fall"]
    )
)

[[104   0   4]
 [  0   2   0]
 [ 11   0  15]]
              precision    recall  f1-score   support

      Normal       0.90      0.96      0.93       108
   Near Fall       1.00      1.00      1.00         2
        Fall       0.79      0.58      0.67        26

    accuracy                           0.89       136
   macro avg       0.90      0.85      0.87       136
weighted avg       0.88      0.89      0.88       136



In [ ]:
classes = ["Normal", "Near Fall", "Fall"]

for i in range(len(test_x)):
    predicted = np.argmax(y_prob[i])
    
    print(
        f"Actual: {classes[test_y[i]]:10} "
        f"Predicted: {classes[predicted]:10} "
        f"Probabilities: {np.round(y_prob[i], 3)}"
    )

Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0. 0.]
Actual: Normal     Predicted: Normal     Probabilities: [1. 0.